# Speech Representations Lab: Comparing Acoustic and Articulatory Representations
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/speech_representations_lab.ipynb)
There are two fundamentally different ways to represent speech:
- **Acoustic representations** decompose speech into perceptual features: pitch (F0),
  loudness, and phonetic content (phonetic posteriorgrams / PPGs). These correspond
  to *what we hear*.
- **Articulatory representations** decompose speech into physical vocal tract
  kinematics using SPARC (Speech Articulatory Coding): tongue, lip, and jaw positions
  plus prosody. These correspond to *how speech is produced*.
In this lab you will:
1. Record (or load) speech samples
2. Extract articulatory features (SPARC EMA, pitch, loudness) and resynthesize speech
3. Extract acoustic features (pitch contour, PPG phoneme timeline)
4. Compare the two representation spaces side by side
5. Explore resynthesis using articulatory features

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab"

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime → Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os
import urllib.request
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio.transforms as T

from senselab.audio.data_structures import Audio
from senselab.audio.tasks.features_extraction.ppg import (
    extract_ppg_segments,
    to_frame_major_posteriorgram,
    extract_mean_phoneme_durations,
    extract_ppgs_from_audios,
    plot_ppg_phoneme_timeline,
)
from senselab.audio.tasks.features_extraction.sparc import SparcFeatureExtractor
from senselab.audio.tasks.features_extraction.torchaudio import extract_pitch_from_audios
from senselab.audio.tasks.plotting.plotting import play_audio, plot_aligned_panels, plot_specgram, plot_waveform
from senselab.audio.tasks.preprocessing.preprocessing import (
    downmix_audios_to_mono,
    resample_audios,
)
from senselab.utils.data_structures import DeviceType

# Auto-detect device
if torch.cuda.is_available():
    device = DeviceType.CUDA
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = DeviceType.CPU  # MPS not supported by all backends
else:
    device = DeviceType.CPU
print(f"Using device: {device.value}")

## Part 1: Recording and Preparation

For a speech/hearing course, everyone should record the same **standard sentence**
so that results can be compared across students. We also load a second recording
from a different speaker for voice conversion later.

The standard sentence below is designed to exercise a wide variety of English
phonemes and articulatory gestures:

> *"The beige ant carried a muffin over the head of Liberty!"*

If you are running in Google Colab with a microphone, you can record your own
voice. Otherwise, sample files are downloaded automatically.

In [ ]:
# Record audio in Google Colab (uses browser microphone)
# If not in Colab, skip this cell and run the next one to load sample files.
from pathlib import Path

Path("tutorial_audio_files").mkdir(exist_ok=True)

try:
    from google.colab import output
    from IPython.display import HTML, display

    RECORD_JS = """
    <button id="record-btn" style="font-size:16px;padding:8px 16px">🎙 Click to Record (5s)</button>
    <script>
    (async () => {
        const btn = document.getElementById('record-btn');
        btn.onclick = async () => {
            btn.textContent = '🔴 Recording...';
            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = async () => {
                stream.getTracks().forEach(t => t.stop());
                const blob = new Blob(chunks, {type: 'audio/wav'});
                const reader = new FileReader();
                reader.onload = () => {
                    const b64 = reader.result.split(',')[1];
                    google.colab.kernel.invokeFunction('save_recording', [b64], {});
                };
                reader.readAsDataURL(blob);
                btn.textContent = '✅ Recorded! Run next cell to load.';
            };
            recorder.start();
            setTimeout(() => recorder.stop(), 5000);
        };
    })();
    </script>
    """

    def save_recording(b64_data):
        import base64

        raw = base64.b64decode(b64_data)
        with open("tutorial_audio_files/my_recording.wav", "wb") as f:
            f.write(raw)

    output.register_callback("save_recording", save_recording)
    display(HTML(RECORD_JS))
    print("Click the button to record 5 seconds of audio.")
    print("After recording, run the NEXT cell to load it.")
except Exception:
    print("Recording not available. Run the next cell to load sample files.")

In [ ]:
from pathlib import Path

# Load recorded audio if available, otherwise download sample files
import urllib.request

base_url = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing"

# Download sample files as fallback
for fname in ["audio_48khz_mono_16bits.wav", "audio_48khz_stereo_16bits.wav"]:
    dest = Path("tutorial_audio_files") / fname
    if not dest.exists():
        urllib.request.urlretrieve(f"{base_url}/{fname}", str(dest))

# Use recording if available, otherwise use sample
rec_path = Path("tutorial_audio_files/my_recording.wav")
if rec_path.exists() and rec_path.stat().st_size > 1000:
    sentence1 = Audio(filepath=str(rec_path.resolve()))
    print("Sentence 1: using your recording")
else:
    sentence1 = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_mono_16bits.wav"))
    print("Sentence 1: using sample audio")

# Second audio for speaker comparison (always sample)
sentence2_raw = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_stereo_16bits.wav"))

# Preprocess: mono + 16kHz
sentence1_16k = resample_audios(downmix_audios_to_mono([sentence1]), resample_rate=16000)[0]
sentence2_16k = resample_audios(downmix_audios_to_mono([sentence2_raw]), resample_rate=16000)[0]

print(f"Sentence 1: {sentence1_16k.waveform.shape[1] / 16000:.2f}s at 16kHz")
print(f"Sentence 2: {sentence2_16k.waveform.shape[1] / 16000:.2f}s at 16kHz (sample, for comparison)")

In [ ]:
print("Sentence 1:")
play_audio(sentence1_16k)

In [ ]:
print("Sentence 2:")
play_audio(sentence2_16k)

In [ ]:
# Waveforms side by side
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=False)

wf1 = sentence1_16k.waveform.squeeze().numpy()
wf2 = sentence2_16k.waveform.squeeze().numpy()
t1 = np.linspace(0, len(wf1) / 16000, len(wf1))
t2 = np.linspace(0, len(wf2) / 16000, len(wf2))

axes[0].plot(t1, wf1, linewidth=0.3, color="steelblue")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Sentence 1")
axes[0].grid(True, alpha=0.2)

axes[1].plot(t2, wf2, linewidth=0.3, color="coral")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("Sentence 2")
axes[1].set_xlabel("Time (seconds)")
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## Part 2: Articulatory (Kinematic) Feature Analysis

[SPARC](https://arxiv.org/abs/2406.12998) (Speech Articulatory Coding) is a
neural encoding-decoding framework that represents speech as **14 articulatory
features** at 50 Hz, plus a disentangled speaker embedding.

Each channel corresponds to a physical articulator on the vocal tract:

| Articulator | Abbreviation | Channels |
|---|---|---|
| Tongue Tip | TT | x, y |
| Tongue Body | TB | x, y |
| Tongue Dorsum | TD | x, y |
| Lower Incisor (jaw) | LI | x, y |
| Upper Lip | UL | x, y |
| Lower Lip | LL | x, y |

In addition, SPARC extracts **pitch** and **loudness** contours, giving a
complete physical description of speech production: *where* the articulators
are and *how* the voice source behaves.

In [ ]:
# Extract SPARC articulatory features
sparc_features = SparcFeatureExtractor.extract_sparc_features(audios=[sentence1_16k], device=device, resample=True)
features1 = sparc_features[0]
print(f"EMA shape: {features1['ema'].shape}  (channels x frames)")
print(f"Pitch shape: {features1['pitch'].shape}")
print(f"Loudness shape: {features1['loudness'].shape}")
print(f"Speaker embedding shape: {features1['spk_emb'].shape}")

### Articulatory Trajectories

The EMA (Electromagnetic Articulography) features show estimated positions of
6 articulators (x and y each = 12 channels). Each trace below represents one
articulator coordinate over time. Notice how the tongue body and tongue tip
move most dynamically during consonant production, while the jaw and lips
show broader, slower movements.

In [ ]:
# Color scheme for articulators
COLOR_CODE = {
    "UL": matplotlib.colors.to_rgb("#EE3A5B"),
    "LL": matplotlib.colors.to_rgb("#FFD155"),
    "LI": matplotlib.colors.to_rgb("#959595"),
    "TT": matplotlib.colors.to_rgb("#43B96B"),
    "TB": matplotlib.colors.to_rgb("#3EADD8"),
    "TD": matplotlib.colors.to_rgb("#B075D0"),
}
CHANNEL_NAMES = [
    "TT_x",
    "TT_y",
    "TB_x",
    "TB_y",
    "TD_x",
    "TD_y",
    "UL_x",
    "UL_y",
    "LL_x",
    "LL_y",
    "LI_x",
    "LI_y",
]

ema = features1["ema"].numpy()
# EMA shape is (frames, channels) — detect orientation
if ema.shape[0] > ema.shape[1]:  # (frames, channels)
    ema = ema.T  # transpose to (channels, frames)
n_channels = min(ema.shape[0], len(CHANNEL_NAMES))
n_frames = ema.shape[1]
time_ema = np.arange(n_frames) / 50.0  # SPARC runs at 50 Hz

fig, ax = plt.subplots(figsize=(14, 8))
offset = 0
for ch in range(n_channels):
    name = CHANNEL_NAMES[ch]
    artic = name.split("_")[0]
    color = COLOR_CODE.get(artic, (0.5, 0.5, 0.5))
    signal = ema[ch] - ema[ch].mean()
    ax.plot(time_ema, signal + offset, color=color, linewidth=0.8, label=name)
    offset += 3

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Articulators (vertically offset)")
ax.set_title("SPARC Articulatory Trajectories (EMA)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### SPARC Pitch and Loudness

SPARC also extracts pitch (fundamental frequency) and loudness contours from
the voice source, providing a prosodic complement to the articulatory features.
Together, these capture the full physical description: *what the articulators do*
and *how the vocal folds vibrate*.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)

pitch_sparc = features1["pitch"].numpy().squeeze()
loudness_sparc = features1["loudness"].numpy().squeeze()
pitch_time = np.arange(len(pitch_sparc)) / 50.0
loud_time = np.arange(len(loudness_sparc)) / 50.0

axes[0].plot(pitch_time, pitch_sparc, linewidth=0.8, color="steelblue")
axes[0].set_ylabel("Pitch")
axes[0].set_title("SPARC Prosody Features")
axes[0].grid(True, alpha=0.3)

axes[1].plot(loud_time, loudness_sparc, linewidth=0.8, color="coral")
axes[1].set_ylabel("Loudness")
axes[1].set_xlabel("Time (seconds)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Articulatory Resynthesis

A key advantage of articulatory representations is that SPARC can **decode**
features back into audio. This demonstrates how well the representation
preserves the speech signal -- if resynthesis sounds natural, the
articulatory encoding is capturing the essential information.

The decoder takes the EMA, pitch, loudness, and speaker embedding and
reconstructs a waveform.

In [ ]:
# Resynthesize speech from articulatory features
resynthesized = SparcFeatureExtractor.decode_sparc_features(features=features1, device=device)
print(f"Resynthesized: {resynthesized.waveform.shape[1] / resynthesized.sampling_rate:.2f}s")

print("\nOriginal:")
play_audio(sentence1_16k)

In [ ]:
print("Resynthesized from articulatory features:")
play_audio(resynthesized)

### Pitch Contour

The pitch (F0) contour shows how the fundamental frequency varies over time.
It carries information about intonation (rising = question, falling = statement),
stress, and emotion.

In [ ]:
# Extract acoustic pitch contour
pitch_result = extract_pitch_from_audios([sentence1_16k], freq_low=80, freq_high=500)
pitch_contour = pitch_result[0]["pitch"].squeeze()

# Build time axis (default hop length for torchaudio pitch detection)
hop_length = 160  # default for 16 kHz
time_pitch = torch.arange(len(pitch_contour)) * hop_length / 16000

# Plot only voiced frames (pitch > 0)
plt.figure(figsize=(12, 4))
voiced = (pitch_contour > 50) & (pitch_contour < 350)  # filter artifacts
plt.scatter(time_pitch[voiced].numpy(), pitch_contour[voiced].numpy(), s=2, c="blue")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")
plt.title("Acoustic Pitch (F0) Contour")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Phonetic Posteriorgram (PPG)

A **PPG** provides the probability of each phoneme at every time frame.
Unlike forced alignment (which gives binary phoneme boundaries), PPGs capture
the continuous, overlapping nature of speech sounds (coarticulation).

The PPG is extracted by a neural network that maps audio frames to phoneme
probability distributions over 40 ARPAbet phonemes.

*Note: PPG extraction uses an isolated subprocess virtual environment. The
first run may take a few minutes to set up.*

In [ ]:
# Extract phonetic posteriorgrams
ppgs = extract_ppgs_from_audios([sentence1_16k], device=device)
ppg1 = ppgs[0]
print(f"PPG shape: {ppg1.shape}")
print("Each column is a time frame; each row is a phoneme probability.")

In [ ]:
# Phoneme duration analysis from PPG
durations = extract_mean_phoneme_durations(sentence1_16k, ppg1)
print(f"Analysis: {durations['frame_count']} frames over {durations['analysis_duration_seconds']:.2f}s")
print("\n=== Top 10 Phonemes by Total Duration ===")
for p in sorted(durations["phoneme_durations"], key=lambda x: x["total_duration_seconds"], reverse=True)[:10]:
    bar = "\u2588" * int(p["total_duration_seconds"] * 50)
    print(f"  {p['phoneme']:>10s}: {p['mean_duration_seconds'] * 1000:6.1f}ms avg ({p['count']} segments) {bar}")

### Time-Aligned Acoustic Visualization

Combining the waveform, spectrogram, pitch overlay, and PPG phoneme segments
on a shared time axis gives a complete acoustic picture of the speech signal.
Each panel shows a different aspect of the same utterance, using `plot_aligned_panels`.

In [ ]:
# Standard spectrogram params: 10ms window, 5ms hop at 16kHz
spec_params = {"n_fft": 256, "hop_length": 80, "win_length": 160}

# Get PPG segments for plotting
frame_major = to_frame_major_posteriorgram(ppg1)
segments = extract_ppg_segments(sentence1_16k, frame_major)

# Convert PPG segments to plot_aligned_panels format
ppg_segments = [
    {"label": seg["phoneme"], "start": seg["start_seconds"], "end": seg["start_seconds"] + seg["duration_seconds"]}
    for seg in segments
]

# Build pitch overlay data
voiced_mask = (pitch_contour > 50) & (pitch_contour < 350)
pitch_data = [(time_pitch[voiced_mask], pitch_contour[voiced_mask], "F0", "steelblue")]

plot_aligned_panels(
    sentence1_16k,
    [
        {"type": "waveform"},
        {"type": "spectrogram", "mel": False},
        {"type": "features", "data": pitch_data, "style": "line"},
        {"type": "segments", "segments": ppg_segments},
    ],
    title="Acoustic Analysis: Waveform + Spectrogram + Pitch + PPG",
    spectrogram_params=spec_params,
    figsize=(16, 14),
)
plt.show()

## Part 4: Comparing Acoustic and Articulatory Representations

Now we have both representations of the same utterance. Let us place them
side by side to understand their similarities and differences.

Key questions to consider:
- Do the pitch contours from SPARC and the acoustic extractor agree?
- How do articulatory trajectories relate to phoneme boundaries?
- Which representation is more intuitive to interpret?

### Side-by-Side Feature Comparison

In [ ]:
# Get PPG segments for the comparison
frame_major = to_frame_major_posteriorgram(ppg1)
segments = extract_ppg_segments(sentence1_16k, frame_major)
ppg_segments = [
    {"label": seg["phoneme"], "start": seg["start_seconds"], "end": seg["start_seconds"] + seg["duration_seconds"]}
    for seg in segments
]

# Build SPARC pitch and acoustic pitch as feature overlays
sparc_pitch_time = np.arange(len(pitch_sparc)) / 50.0
voiced_mask = (pitch_contour > 50) & (pitch_contour < 350)

pitch_comparison_data = [
    (sparc_pitch_time, pitch_sparc, "SPARC pitch", "coral"),
    (time_pitch[voiced_mask], pitch_contour[voiced_mask], "Acoustic F0", "steelblue"),
]

loudness_data = [(loud_time, loudness_sparc, "SPARC Loudness", "coral")]

# Selected EMA trajectories
selected_channels = [("TT_y", 1), ("TB_y", 3), ("UL_y", 7), ("LL_y", 9)]
ema_data = []
for name, ch_idx in selected_channels:
    artic = name.split("_")[0]
    color_map = {
        "TT": "#43B96B",
        "TB": "#3EADD8",
        "UL": "#EE3A5B",
        "LL": "#FFD155",
    }
    signal = ema[ch_idx] - ema[ch_idx].mean()
    ema_data.append((time_ema, signal, name, color_map.get(artic, "gray")))

plot_aligned_panels(
    sentence1_16k,
    [
        {"type": "waveform"},
        {"type": "features", "data": pitch_comparison_data, "style": "line"},
        {"type": "features", "data": loudness_data, "style": "line"},
        {"type": "features", "data": ema_data, "style": "line"},
        {"type": "segments", "segments": ppg_segments},
    ],
    title="Acoustic vs Articulatory Representations (side by side)",
    spectrogram_params=spec_params,
    figsize=(16, 18),
)
plt.show()

### Questions for Analysis

Examine the comparison plot above and your earlier results to answer
the following questions:

1. **Pitch**: Compare the SPARC pitch contour (Part 2) with the acoustic
   pitch contour (Part 3). Where do they agree? Where do they diverge?
   What might explain the differences?

2. **Interpretability**: If you wanted to explain to someone *what* was
   said, which representation would you use? If you wanted to explain
   *how* it was produced physically, which would you choose? Why?

3. **Independence**: Suppose you wanted to raise the pitch of the
   utterance without changing the words. How would you do this in the
   acoustic representation? How would you do it in the articulatory
   representation? Which approach risks unintended side effects?

4. **Content modification**: Now suppose you wanted to change a specific
   phoneme (e.g., turn a /t/ into a /d/). What would you modify in each
   representation? Which seems more straightforward?

5. **Resynthesis**: Listen to the original and SPARC-resynthesized audio
   from Part 2. What is preserved? What is lost or changed? What does
   this tell you about how complete the articulatory representation is?

6. **Neural control**: Based on your observations, which representation
   space would be easier for a neural system to learn to control? Justify
   your answer with specific evidence from your analysis.


### Comparison Summary

In [ ]:
print("""
| Aspect              | Acoustic (PPG/Pitch/Loudness) | Articulatory (SPARC)        |
|---------------------|-------------------------------|-----------------------------||
| Space               | Perceptual                    | Physical (vocal tract)      |
| Pitch               | Explicit F0 contour           | Source F0 parameter         |
| Content             | Phoneme probabilities (PPG)   | Articulator positions (EMA) |
| Independence        | Disentangled                  | Interdependent              |
| Edit pitch          | Modify F0 curve directly      | Modify source F0            |
| Change phoneme      | Edit PPG labels               | Adjust articulator paths    |
| Resynthesis         | Vocoder needed                | Built-in decoder            |
| Voice conversion    | Separate model needed         | Built-in converter          |
""")

## Summary

In this lab we decomposed the same speech signal using two complementary
approaches:

| Representation | Features | senselab API |
|---|---|---|
| **Acoustic** | Pitch (F0) contour | `extract_pitch_from_audios()` |
| **Acoustic** | Phonetic Posteriorgrams (PPG) | `extract_ppgs_from_audios()` |
| **Articulatory** | EMA articulator positions | `SparcFeatureExtractor.extract_sparc_features()` |
| **Articulatory** | SPARC pitch + loudness | (included in SPARC output) |
| **Articulatory** | Resynthesis from features | `SparcFeatureExtractor.decode_sparc_features()` |
| **Articulatory** | Voice conversion | `SparcFeatureExtractor.convert_voice()` |

Use your observations from the questions above to formulate your own
conclusions about which representation space is more appropriate for
neural control of speech, and why. Consider the trade-offs between
perceptual interpretability, physical grounding, independence of
features, and ease of modification.
